# EDA — Dengue grave (COD_EVE 220) · 2007–2024

Análisis exploratorio del dataset consolidado de dengue grave. Cubre calidad de datos, serie temporal, geografía, indicadores de severidad (hospitalización, mortalidad) y comparación de tasas de gravedad respecto al dengue clásico.

**Prerrequisito:** haber corrido `03_Combinar_SIVIGILA_Dengue_Grave.ipynb`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4),
                     'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
CSV_GRAVE   = "../data/processed/sivigila_dengue_grave_consolidado.csv"
CSV_CLASICO = "../data/processed/sivigila_dengue_consolidado.csv"   # para comparación

EPIDEMIC_YEARS = [2010, 2013, 2016, 2019, 2023, 2024]

## 1. Carga y vista general

In [ ]:
df = pd.read_csv(CSV_GRAVE, dtype=str, low_memory=False)
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head(3)

## 2. Calidad de datos

In [ ]:
# Nulos por columna (solo las que tienen al menos 1%)
nulos = df.isnull().mean().mul(100).round(1)
nulos = nulos[nulos > 1].sort_values(ascending=False)
print("Columnas con >1% nulos:")
display(nulos.to_frame(name="% nulos"))

In [ ]:
# Duplicados por CONSECUTIVE
dup = df['CONSECUTIVE'].duplicated().sum()
print(f"Duplicados por CONSECUTIVE: {dup:,}")

# Distribución de TIP_CAS — en 220 todos deberían ser graves
print("\nDistribución TIP_CAS:")
display(df['TIP_CAS'].value_counts().to_frame())

## 3. Serie temporal — casos por año

In [ ]:
df['ANO'] = pd.to_numeric(df['ANO'], errors='coerce')
por_ano = df.groupby('ANO').size().reset_index(name='casos')

fig, ax = plt.subplots()
colors = ['#BE1D2B' if y in EPIDEMIC_YEARS else '#5C8DBE' for y in por_ano['ANO']]
bars = ax.bar(por_ano['ANO'], por_ano['casos'], color=colors, width=0.7)
ax.set_xlabel('Año')
ax.set_ylabel('Casos de dengue grave')
ax.set_title('Casos de dengue grave por año (SIVIGILA 2007–2024)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#BE1D2B', label='Año epidémico'),
                   Patch(color='#5C8DBE', label='Año normal')])
plt.tight_layout()
plt.show()

display(por_ano.set_index('ANO').style.format({'casos': '{:,}'}))

## 4. Tasa de gravedad respecto al dengue clásico

In [ ]:
df_c = pd.read_csv(CSV_CLASICO, usecols=['ANO'], dtype=str)
df_c['ANO'] = pd.to_numeric(df_c['ANO'], errors='coerce')

clasico_ano = df_c.groupby('ANO').size().rename('clasico')
grave_ano   = df.groupby('ANO').size().rename('grave')

tasa = pd.concat([clasico_ano, grave_ano], axis=1).dropna()
tasa['tasa_%'] = (tasa['grave'] / tasa['clasico'] * 100).round(2)

fig, ax = plt.subplots()
ax.plot(tasa.index, tasa['tasa_%'], marker='o', color='#BE1D2B', linewidth=2)
ax.axhline(tasa['tasa_%'].mean(), linestyle='--', color='gray', label=f"Media {tasa['tasa_%'].mean():.1f}%")
ax.set_xlabel('Año')
ax.set_ylabel('Tasa de gravedad (%)')
ax.set_title('Tasa de gravedad = casos graves / casos clásicos por año')
ax.legend()
plt.tight_layout()
plt.show()

display(tasa.style.format({'clasico': '{:,}', 'grave': '{:,}', 'tasa_%': '{:.2f}%'}))

## 5. Estacionalidad — casos por semana epidemiológica

In [ ]:
df['SEMANA'] = pd.to_numeric(df['SEMANA'], errors='coerce')
df_ref = df[~df['ANO'].isin(EPIDEMIC_YEARS)]

sem_stats = df_ref.groupby('SEMANA').size().rename('casos')

fig, ax = plt.subplots()
ax.bar(sem_stats.index, sem_stats.values, color='#5C8DBE', width=0.8)
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos promedio')
ax.set_title('Estacionalidad dengue grave (años no epidémicos)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 6. Geografía — top municipios y departamentos

In [ ]:
top_mun = df['Municipio_ocurrencia'].value_counts().head(20)
top_dpto = df['Departamento_ocurrencia'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_mun.sort_values().plot.barh(ax=axes[0], color='#BE1D2B')
axes[0].set_title('Top 20 municipios — dengue grave')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

top_dpto.sort_values().plot.barh(ax=axes[1], color='#5C8DBE')
axes[1].set_title('Top 10 departamentos — dengue grave')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

## 7. Indicadores de severidad — hospitalización y mortalidad

In [ ]:
# PAC_HOS: 1=Sí hospitalizado, 2=No
# CON_FIN: 1=Vivo, 2=Muerto, 3=No sabe
hosp = df['PAC_HOS'].value_counts(normalize=True).mul(100).round(1)
outcome = df['CON_FIN'].value_counts(normalize=True).mul(100).round(1)

print("Hospitalización (PAC_HOS):")
display(hosp.to_frame(name='%'))

print("\nDesenlace (CON_FIN):")
display(outcome.to_frame(name='%'))

In [ ]:
# Mortalidad por año
df['CON_FIN'] = pd.to_numeric(df['CON_FIN'], errors='coerce')
muertes_ano = df[df['CON_FIN'] == 2].groupby('ANO').size().rename('muertes')
tasa_mort = pd.concat([grave_ano, muertes_ano], axis=1).fillna(0)
tasa_mort['letalidad_%'] = (tasa_mort['muertes'] / tasa_mort['grave'] * 100).round(2)

fig, ax = plt.subplots()
ax.bar(tasa_mort.index, tasa_mort['muertes'], color='#BE1D2B', width=0.7)
ax2 = ax.twinx()
ax2.plot(tasa_mort.index, tasa_mort['letalidad_%'], marker='o',
         color='#1B2233', linewidth=2, label='Letalidad %')
ax.set_xlabel('Año')
ax.set_ylabel('Muertes')
ax2.set_ylabel('Letalidad (%)')
ax.set_title('Muertes y tasa de letalidad por dengue grave 2007–2024')
ax2.legend(loc='upper left')
plt.tight_layout()
plt.show()

display(tasa_mort.style.format({'grave': '{:,}', 'muertes': '{:,}', 'letalidad_%': '{:.2f}%'}))